In [1]:
import pathlib
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import datetime
import sys
import os
from copy import deepcopy
import torch
from torch.optim.lr_scheduler import OneCycleLR

In [2]:
# add cnrm code to python path
sys.path.append('/home/users/jar212/ai_downscaling_fork/src/cnrm-unet/src')

from data import CORDEXDataset
from model import UNet
from loss import fit_gamma_distributions, EmulASYMLoss
from train import train_loop, test_loop
from visualisation import log_prediction_visualisation

In [3]:
# Path to CORDEX-Bench dataset
cordexbench_root = pathlib.Path('/gws/nopw/j04/mohc_shared/cordexbench')
print(cordexbench_root.is_dir())

True


In [4]:
# Load predictor dataset
predictor_filename = f'{cordexbench_root}/SA_domain/train/Emulator_hist_future/predictors/ACCESS-CM2_1961-1980_2080-2099.nc'
predictor = xr.open_dataset(predictor_filename)
predictor = predictor.drop_vars('time_bnds')

In [5]:
# Load the predictands (target) dataset
predictand_filename = f'{cordexbench_root}/SA_domain/train/Emulator_hist_future/target/pr_tasmax_ACCESS-CM2_1961-1980_2080-2099.nc'
predictand = xr.open_dataset(predictand_filename)
predictand = predictand.drop_vars(['time_bnds', 'lon_bnds', 'lat_bnds', 'crs']) # SA dataset contains these additional variables which can be dropped

In [6]:
# Function to save time slice of dataset
def save_year_slice(ds, start_year, end_year, gcm_name, out_dir, is_predictand=False):
    # Make directory
    out_dir.mkdir(parents=True, exist_ok=True)

    # Select years
    years = range(start_year, end_year + 1)
    ds_slice = ds.sel(time=ds.time.dt.year.isin(years))

    # Add prefix if dataset is a predictand
    prefix = 'pr_tasmax_' if is_predictand else ''

    # Construct filename
    filename = f'{prefix}{gcm_name}_{start_year}-{end_year}.nc'
    out_file = out_dir / filename

    # Save slice
    ds_slice.to_netcdf(out_file)
    print(f'Saved: {out_file}')
    
    return out_file

In [7]:
# Specify training and validation period
training_years = list(range(1961,1977))
validation_years = list(range(1977,1979))

In [8]:
# Location to save time slice of predictors and predictands
user_dir = pathlib.Path('/gws/nopw/j04/mohc_shared/users/jar212/')
predictor_dir = user_dir / 'cordexbench' / 'SA_domain' / 'predictors'
predictand_dir = user_dir / 'cordexbench' / 'SA_domain' / 'target'

save_year_slice(predictor, 1961, 1976, 'ACCESS-CM2', predictor_dir)
save_year_slice(predictor, 1977, 1978, 'ACCESS-CM2', predictor_dir)
save_year_slice(predictand, 1961, 1976, 'ACCESS-CM2', predictand_dir, is_predictand=True)
save_year_slice(predictand, 1977, 1978, 'ACCESS-CM2', predictand_dir, is_predictand=True)


Saved: /gws/nopw/j04/mohc_shared/users/jar212/cordexbench/SA_domain/predictors/ACCESS-CM2_1961-1976.nc
Saved: /gws/nopw/j04/mohc_shared/users/jar212/cordexbench/SA_domain/predictors/ACCESS-CM2_1977-1978.nc
Saved: /gws/nopw/j04/mohc_shared/users/jar212/cordexbench/SA_domain/target/pr_tasmax_ACCESS-CM2_1961-1976.nc
Saved: /gws/nopw/j04/mohc_shared/users/jar212/cordexbench/SA_domain/target/pr_tasmax_ACCESS-CM2_1977-1978.nc


PosixPath('/gws/nopw/j04/mohc_shared/users/jar212/cordexbench/SA_domain/target/pr_tasmax_ACCESS-CM2_1977-1978.nc')

In [12]:
# Path to the output directory. All outputs will be saved here (model, normalisation stats)
output_dir_path = user_dir / 'cordexbench' / 'SA_domain' / 'outputs'
output_dir_path.mkdir(parents=True, exist_ok=True)

# Path to the predictors training dataset
training_predictors_path = user_dir / 'cordexbench' / 'SA_domain' / 'predictors' / 'ACCESS-CM2_1961-1976.nc'

# Path to the target training dataset
training_target_path = user_dir / 'cordexbench' / 'SA_domain' / 'target' / 'pr_tasmax_ACCESS-CM2_1961-1976.nc'

# Path to the predictors validation dataset
validation_predictors_path = user_dir / 'cordexbench' / 'SA_domain' / 'predictors' / 'ACCESS-CM2_1977-1978.nc'

# Path to the target validation dataset
validation_target_path = user_dir / 'cordexbench' / 'SA_domain' / 'target' / 'pr_tasmax_ACCESS-CM2_1977-1978.nc'

# Selection of predictors
predictors_selection = ['t_850','z_500','q_850']

# Selection of targets
targets_selection = ['pr']

# Reference period start
reference_period_start = 1961

# Reference period end
reference_period_end = 2100

# Factor to multiply target by. This may be used to convert from units within the dataset to the desired units
factor = 1

# Loss function. Must be one of 'mse', 'mae', 'emulasym'
loss = 'mse'

# Number of epochs to train for
epochs = 10

# Batch size
batch_size = 32

# Base number of channels in the model
model_channels = 64

# List of channel multipliers
channel_mult = [1, 2, 4, 8, 8]

# Input resolution
input_resolution = 16

# Output resolution
output_resolution = 128

# Optimiser type: 'adam' or 'sgd'
optimiser_type = 'adam'

# Learning rate for the optimiser (default: 5e-4)
learning_rate = 5e-4

# Scheduler type: 'onecycle' or 'steplr' (default: onecycle)
scheduler_type = 'onecycle'

# Step size for StepLR scheduler (default: 10)
scheduler_step_size = 10

# Gamma for StepLR scheduler (default: 0.1)
scheduler_gamma = 0.1

# Max LR for OneCycleLR scheduler (default: 5e-4)
scheduler_max_lr = 5e-4

In [13]:
# Normalisation stats will be saved here
normalisation_stats_path = f"{output_dir_path}/normalisation_stats.json"

In [14]:
# Infer number of output_channels from number of targets
output_channels = len(targets_selection)

In [15]:
# Load training dataset
training_dataset = CORDEXDataset(
    predictors_data_path=training_predictors_path,
    target_data_path=training_target_path,
    predictors=predictors_selection,
    targets=targets_selection,
    normalisation_stats_path=normalisation_stats_path,
    is_train=True,
    reference_period_start=reference_period_start,
    reference_period_end=reference_period_end,
    factor=factor,
)

In [16]:
training_dataset.normalised_two_d_predictors.shape, training_dataset.target_field.shape

((5840, 3, 16, 16), (5840, 128, 128, 1))

In [17]:
# Load validation dataset
validation_dataset = CORDEXDataset(
        predictors_data_path=validation_predictors_path,
        target_data_path=validation_target_path,
        predictors=predictors_selection,
        targets=targets_selection,
        normalisation_stats_path=normalisation_stats_path,
        is_train=False,
        reference_period_start=reference_period_start,
        reference_period_end=reference_period_end,
        factor=factor,
    )

In [18]:
validation_dataset.normalised_two_d_predictors.shape, validation_dataset.target_field.shape

((730, 3, 16, 16), (730, 128, 128, 1))

In [19]:
training_dataloader = torch.utils.data.DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
validation_dataloader = torch.utils.data.DataLoader(validation_dataset, batch_size=batch_size)

In [20]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [21]:
model = UNet(
    num_2d_predictors=training_dataset.num_2d_predictors,
    num_1d_predictors=training_dataset.num_1d_predictors,
    model_channels=model_channels,
    channel_mult=channel_mult,
    input_resolution=input_resolution,
    output_resolution=output_resolution,
    output_channels=output_channels,
).to(DEVICE)

In [22]:
if loss == "mse":
    loss_fn = torch.nn.MSELoss()
elif loss == "mae":
    loss_fn = torch.nn.L1Loss()
elif loss == "emulasym":
    if "pr" not in targets or len(targets) > 1:
        raise ValueError(
            "EmulASYM loss function should only be used for predicting precipitation (pr)."
        )

    # Fit gamma distributions to the training data
    logger.info("Fitting gamma distributions to the training data.")
    alphas, betas = fit_gamma_distributions(
        training_dataset.target_field,
        training_dataset.time,
        reference_period_start,
        reference_period_end,
    )

    np.save(f"{output_dir_path}/alphas.npy", alphas)
    np.save(f"{output_dir_path}/betas.npy", betas)

    alphas = torch.tensor(alphas, dtype=torch.float32).to(DEVICE)
    betas = torch.tensor(betas, dtype=torch.float32).to(DEVICE)

    loss_fn = EmulASYMLoss(alphas, betas)
else:
    raise ValueError(
        f"Invalid loss function '{loss}'. Must be one of 'mse', 'mae', 'emulasym'."
    )


In [23]:
# Optimiser selection
if optimiser_type.lower() == "adam":
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
elif optimiser_type.lower() == "sgd":
    optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)
else:
    raise ValueError(f"Unsupported optimiser: {optimiser_type}")

In [24]:
# Scheduler selection
if scheduler_type.lower() == "onecycle":
    scheduler = OneCycleLR(
        optimiser,
        max_lr=scheduler_max_lr,
        steps_per_epoch=len(training_dataloader),
        epochs=epochs,
    )
elif scheduler_type.lower() == "steplr":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimiser,
        step_size=scheduler_step_size,
        gamma=scheduler_gamma,
    )
else:
    raise ValueError(f"Unsupported scheduler: {scheduler_type}")

In [25]:
%%time
best_val_loss = np.inf
for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_loop(
        training_dataloader, model, loss_fn, optimiser, scheduler, DEVICE, epoch, scheduler_type
    )

    val_loss = test_loop(validation_dataloader, model, loss_fn, DEVICE, epoch)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())

    if scheduler is not None:
        print(f"Current learning rate: {scheduler.get_last_lr()[0]}")

    log_prediction_visualisation(model, validation_dataset, DEVICE, epoch, predictors_selection, targets_selection)

Epoch 1
-------------------------------


2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/27 17:37:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/27 17:38:39 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/27 17:38:39 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Current learning rate: 0.00014039740120325237
Epoch 2
-------------------------------
Current learning rate: 0.0003807934863422019
Epoch 3
-------------------------------
Current learning rate: 0.0004999992481874969
Epoch 4
-------------------------------
Current learning rate: 0.0004749756200778733
Epoch 5
-------------------------------
Current learning rate: 0.00040539300932918
Epoch 6
-------------------------------
Current learning rate: 0.00030503310537491357
Epoch 7
-------------------------------
Current learning rate: 0.00019377341802666507
Epoch 8
-------------------------------
Current learning rate: 9.365029287427422e-05
Epoch 9
-------------------------------
Current learning rate: 2.4494342783719323e-05
Epoch 10
-------------------------------
Current learning rate: 2.7518125031682012e-09


In [ ]:
model_path = f"{output_dir_path}/model.pth"
print(f"Training complete. Saving model to {model_path}")
torch.save(best_model_state, model_path)